*Your Name*

*Collaborator's Names*

# Overall Title

My goals for today:
* get a better understanding of how Pandas works under the hood, esp as pertains to `apply`
* glimpse the tradeoff between performance and readability with specialized v generalized functions

## Dataset Introspection

The dataset `customer_hours.csv` contains a large amount (100,000 rows) of fake data representing customer cell phone call time over a two week period. The first column is the customer's unqiue ID number and each subsequent column is the number of hours the customer has spent on a call that day.

The datasets `species_data_complete.csv` and `species_data_incomplete.csv` are smaller datasets (100 rows) of fake bird observations reported by the public. For both files, the first column is an ID for the observer and each subsequent column counts the number of individuals seen for each species. In `species_data_incomplete.csv`, however, the data has empty values mixed in when members of the public fail to report data for a particular species.

In [48]:
import pandas as pd

df_ctime = pd.read_csv("customer_data.csv", index_col="CustomerID")
df_birds_cmp = pd.read_csv("species_data_complete.csv", index_col="observer")
df_birds_inc = pd.read_csv("species_data_incomplete.csv", index_col="observer")

We can confirm that Pandas is using NumPy's datatypes for its columns by comparing a column's `dtype` attribute to the NumPy type:

In [49]:
import numpy as np
print("Index stored using NumPy integer:", df_ctime.index.dtype == np.int64)
print("Data stored using NumPy float:", df_ctime["05/01/23"].dtype == np.float64)

Index stored using NumPy integer: True
Data stored using NumPy float: True


---
### Exercises

1. If we don't specify the data type (`dtype`) when loading a file, Pandas will automatically guess the best choice. Using the two bird datasets loaded above, what is Pandas's `dtype` choice for data with and without missing values?

Note: DataFrames have the `dtypes` attribute while individual columns have `dtype`. You can answer this question using either attribute.

In [50]:
print(df_birds_cmp["bird02"].dtype)
print(df_birds_inc["bird02"].dtype)

int64
float64


*Answer here*

2. Why do you think Pandas make this choice?

*Answer here*

3. We risk losing some precision if we represent integer numbers with `float64`; however, NumPy's `int64` data type doesn't support missing values by itself (and neither does the `bool` type). As discussed in lecture, Pandas offers the `Int64` type (notice the capitalization) that combines an `int64` NumPy array with a Boolean mask to indicate missing values. [Read more about this type](https://pandas.pydata.org/docs/user_guide/integer_na.html) and re-load `species_data_incomplete.csv` to use it; then, confirm it is **not** the same type as NumPy's `int64` following the explanation at the beginning of this section.

In [51]:
df_birds_inc_copy = pd.read_csv("species_data_incomplete.csv",
                                index_col="observer",
                                dtype="Int64")
print(df_birds_inc_copy["bird02"].dtype == np.int64)

False


4. Though `Int64` uses a second array to track missing values, this array doesn't need much space in memory. The two states -- data missing and data not missing -- can be described by either a 1 or a 0. This means each `Int64` value is represented by 65 bits: 64 for the integer itself and one for whether or not the value is considered missing. How does this compare to the number of bits needed to store `float64`?

*Answer here*

5. Last week we saw that transferring data to the CPU impacts our performance. Do you think using `Int64` instead of `float64` adds enough extra data to make an appreciable impact on performance?

*Answer here*

6. Time how long it takes to sum up the total number of birds seen in `species_data_incomplete.csv` when using both the `float64` and `Int64` data types. How do the performance measurements align with your expectations in Question 5?

In [52]:
%timeit df_birds_inc.sum()
%timeit df_birds_inc_copy.sum()

48.1 ms ± 952 µs per loop (mean ± std. dev. of 7 runs, 10 loops each)
43.8 ms ± 509 µs per loop (mean ± std. dev. of 7 runs, 10 loops each)


*Answer here*

---
## Operating on Rows

A common operation with Pandas DataFrames is to operate on each row. As an example, we'll apply ordinary least squares (OLS) to fit a line $y=mx+b$ to the fake cell phone data from before (`customer_data.csv`, which we loaded into the `df_ctime` DataFrame). For this example, we are only interested in how cell phone usage changes over time, so we only need the slope $m$ from the OLS fit.

There are two parts to this task where we can explore performance: first, we need a function for performing OLS. Second, we need to apply this function to each row in the DataFrame.

### Ordinary Least Squares

OLS is an incredibly common algorithm, so much so that NumPy, SciPy, and scikit-learn all have their own implementations. How do they compare against each other? Using the following code as setup, you'll write and test a "wrapper" for NumPy's [`lstsq`](https://numpy.org/doc/stable/reference/generated/numpy.linalg.lstsq.html), SciPy's [`linregress`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.linregress.html), and scikit-learn's [`LinearRegression` class](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html).

In [53]:
from numpy.linalg import lstsq 
from scipy.stats import linregress
from sklearn.linear_model import LinearRegression

test_row = df_ctime.iloc[0]

**The resulting slope should be around -0.0117.**

---
### Exercises

1. Write wrapper functions for each of the routines named above. Each wrapper should take a row from a Pandas DataFrame as input and return the slope of the fitted line. Reference the documentation linked above as needed. 

Notes:
* For simplicity, you may use integers to represent the days (the independent variable) rather than the calendar date in the column label; e.g.,  `x = np.arange(row.shape[0])`. 
* Even if you don't fully understand what NumPy's `lstsq` is doing, the example matches well to our problem.
* scikit-learn is intended to be used for machine learning. The documentation references values `n_features` and `n_targets`; for our purposes, these are each 1. Additionally, the `y` value for the `fit` method is the unmodified data from `row`.

In [55]:
def ols_scipy(row):
    x = np.arange(row.shape[0])
    m, b, r, p, e = linregress(x, row)
    return m

In [54]:
def ols_numpy(row):
    x = np.arange(row.shape[0])
    ones = np.ones(row.shape[0])
    A = np.vstack((x, ones)).T
    m, b = lstsq(A, row)[0]
    return m

In [56]:
def ols_sklearn(row):
    est = LinearRegression() 
    X = np.arange(row.shape[0]).reshape(-1, 1) # shape (14, 1)
    est.fit(X, row) 
    m = est.coef_[0]
    return m

2. Time each function using `test_row`. Which method is fastest? Which is slowest?

In [57]:
%timeit ols_numpy(test_row)
%timeit ols_scipy(test_row)
%timeit ols_sklearn(test_row)


/tmp/ipykernel_99388/434929721.py:5: FutureWarning: `rcond` parameter will change to the default of machine precision times ``max(M, N)`` where M and N are the input matrix dimensions.
To use the future default and silence this warning we advise to pass `rcond=None`, to keep using the old, explicitly pass `rcond=-1`.
  m, b = lstsq(A, row)[0]


47.5 µs ± 669 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)
96.2 µs ± 15.7 µs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)
381 µs ± 6.31 µs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [61]:
ols_numpy(test_row), ols_scipy(test_row), ols_sklearn(test_row)

/tmp/ipykernel_99388/434929721.py:5: FutureWarning: `rcond` parameter will change to the default of machine precision times ``max(M, N)`` where M and N are the input matrix dimensions.
To use the future default and silence this warning we advise to pass `rcond=None`, to keep using the old, explicitly pass `rcond=-1`.
  m, b = lstsq(A, row)[0]


(-0.011721611721611687, -0.011721611721611716, -0.011721611721611718)

*Answer here*

3. SciPy *also* offers a `lstsq` function in it's linear algebra module: `scipy.linalg.lstsq`. In fact, SciPy offers a number of ways to tackle our linear fitting problem. SciPy's `lstsq` behaves very similarly to NumPy's `lstsq`. Create a new wrapper for SciPy's `lstsq` based on your NumPy function.

In [58]:
def ols_scipy2(row):
    x = np.arange(row.shape[0])
    ones = np.ones(row.shape[0])
    A = np.vstack((x, ones)).T
    m, c = sp_lstsq(A, row)[0]
    return m

4. Time the SciPy `lstsq` wrapper. How does it compare to the NumPy version? Using Google, can you find an explanation of the difference?

In [59]:
%timeit ols_scipy2(test_row)

60 µs ± 2.95 µs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


According to [this stack overflow entry](https://stackoverflow.com/questions/29372559/what-is-the-difference-between-numpy-linalg-lstsq-and-scipy-linalg-lstsq), NumPy uses a faster LAPACK routine (but possibly more memory).

5. Which was the *easiest* OLS version to implement? How does the performance of this version compare to the others?

*Answer here*

6. In your opinion, which OLS version represents the best compromise between performance and ease of use? When thinking about "ease of use," consider not only your own time spent developing but *also* the readability of your code for others (including your future self).

*Answer here*

7. Let's find out why scikit-learn had the slowest OLS implementation.

## aksdfah

We'll compare several ways of accomplishing this, starting with the most "purely Pythonic" approach and progressively leveraging more of Pandas's features.